# Visual Question Answering (VQA v2) - Run v4
## Pipeline Overview
This notebook executes the v4 experiment pipeline:
1. **Restore v3 artifacts**: Search /kaggle/input for cached features, vocabulary, and v3 checkpoints.
2. **Feature Extraction**: Resume or verify ResNet-50 feature extraction (train_img_features.h5, val_img_features.h5).
3. **Resume VQA (mul)**: Resume training from v3 checkpoint up to 40 epochs with ReduceLROnPlateau and patience 5.
4. **Resume Question-Only**: Resume question-only baseline up to 40 epochs with ReduceLROnPlateau and patience 5.
5. **Concat Fusion Ablation**: Train VQA model with concatenation fusion from scratch for 15 epochs (ADR-008).
6. **Evaluation**: Evaluate all available checkpoints on the validation set and generate metrics tables and learning curves.
7. **Inference Demo and Summary**: Test predict function and display generated outputs.

In [ ]:
import os
from pathlib import Path

print("=== Checking available datasets in /kaggle/input ===")
base_path = Path("/kaggle/input")
if base_path.exists():
    for root, dirs, files in os.walk(base_path):
        depth = len(Path(root).relative_to(base_path).parts)
        if depth <= 2:
            print(f"{'  ' * depth}[DIR] {Path(root).name}/ ({len(files)} files, {len(dirs)} subdirs)")
            for f in files[:3]:
                print(f"{'  ' * depth}   - {f}")
else:
    print("/kaggle/input directory not found. Not running inside Kaggle environment.")

In [ ]:
# Clone repository and install dependencies
!rm -rf /kaggle/working/science
!git clone https://github.com/TryHanger/science.git /kaggle/working/science
%cd /kaggle/working/science

import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

!pip install -q -r requirements.txt
print("[*] Repository cloned, current working directory:", os.getcwd())

In [ ]:
import os
import shutil
from pathlib import Path

outputs_dir = Path("/kaggle/working/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

input_path = Path("/kaggle/input")
artifact_dir = None

if input_path.exists():
    for p in input_path.rglob("best_model.pth"):
        if p.is_file():
            artifact_dir = p.parent
            break

target_files = [
    "train_img_features.h5",
    "val_img_features.h5",
    "vocab.json",
    "best_model.pth",
    "best_question_only_model.pth",
    "metrics_history_vqa.json",
    "metrics_history_question_only.json",
]

if artifact_dir is not None:
    print(f"[*] Found artifacts directory: {artifact_dir}")
    copied = []
    for fname in target_files:
        src = artifact_dir / fname
        dst = outputs_dir / fname
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / (1024 * 1024)
            copied.append((fname, size_mb))
        elif dst.exists():
            print(f"  [-] Already exists in outputs: {fname}")
    if copied:
        print("[*] Restored artifacts:")
        for fname, size_mb in copied:
            print(f"  + {fname} ({size_mb:.2f} MB)")
    else:
        print("  [*] No new files copied.")
else:
    print("[!] WARNING: Artifact directory with best_model.pth not found under /kaggle/input. Proceeding from scratch.")

In [ ]:
!python src/extract_features.py --env kaggle

In [ ]:
!python src/train.py --env kaggle --model-type vqa --resume --epochs 40 --scheduler plateau --patience 5

In [ ]:
!python src/train.py --env kaggle --model-type question_only --resume --epochs 40 --scheduler plateau --patience 5

In [ ]:
!python src/train.py --env kaggle --model-type vqa --fusion concat --epochs 15 --scheduler none --patience 3

In [ ]:
!python src/evaluate.py --env kaggle

In [ ]:
from pathlib import Path
from src.config import load_config
from src.predict import predict

try:
    val_dir = load_config(env="kaggle").data.val_img_dir
    val_images = list(Path(val_dir).glob("*.jpg"))
    if val_images:
        sample_img = str(val_images[0])
        ans, conf = predict(sample_img, "is the picture colorful?", env="kaggle")
        print(f"Sample image: {sample_img}")
        print(f"Prediction:   {ans} (confidence: {conf:.2%})")
    else:
        print(f"[!] Warning: No images found in validation directory: {val_dir}")
except Exception as e:
    print(f"[!] Prediction demo failed with error: {e}")

In [ ]:
from pathlib import Path

outputs_dir = Path("/kaggle/working/outputs")
print("=== Outputs Directory Summary ===")
if outputs_dir.exists():
    for f in sorted(outputs_dir.iterdir()):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"{f.name:35s} : {size_mb:8.2f} MB")
        elif f.is_dir():
            print(f"{f.name:35s} : [DIR]")
else:
    print(f"Outputs directory not found: {outputs_dir}")

metrics_file = outputs_dir / "metrics_table.csv"
print("\n=== Metrics Table (metrics_table.csv) ===")
if metrics_file.exists():
    print(metrics_file.read_text(encoding="utf-8"))
else:
    print(f"Metrics table not found at {metrics_file}")